In [313]:
import numpy as np
from mapping_tese.modules import constants as C

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

SCHEME = "s1_hrf6_win1"
CACHE_DIR = C.PROJECT_ROOT / "data" / "extracted" / SCHEME
#window or lss(single-trial betas)
FEATURE_METHOD = "lss"

subject = "sub-p0004"

### Load cached features for subject

In [314]:
from mapping_tese.modules.cache import load_subject_samples, cache_path
from mapping_tese.modules.paths import resolve_bold_path
from nilearn.image import load_img, index_img
import joblib

dataset_path = cache_path(CACHE_DIR, FEATURE_METHOD, subject)
masker_path = CACHE_DIR / f"{subject}_masker.joblib"
if not dataset_path.exists() or not masker_path.exists():
    raise FileNotFoundError(f"Missing cached features for {subject}. Run mapping_tese.modules.build_features first.")

X_features, y, groups, tr_subject = load_subject_samples(dataset_path)
masker = joblib.load(masker_path)
run1_path = resolve_bold_path(subject, run=1, space_name=C.SPACE)
ref_img = index_img(load_img(str(run1_path)), 0)

unique_classes = np.unique(y)
chance_level = 1.0 / len(unique_classes)
print(f"Feature matrix: {X_features.shape}")
print(f"Classes: {list(unique_classes)}")
print(f"Samples/class: { {c: int(np.sum(y == c)) for c in unique_classes} }")

Feature matrix: (160, 2187)
Classes: [0, 1, 2, 3]
Samples/class: {0: 24, 1: 32, 2: 48, 3: 56}


In [315]:
from pathlib import Path

RESULTS_DIR = Path("results")
subject_results_dir = RESULTS_DIR / FEATURE_METHOD / subject
subject_models_dir = subject_results_dir / "models"
subject_results_dir.mkdir(parents=True, exist_ok=True)
subject_models_dir.mkdir(parents=True, exist_ok=True)

In [316]:
#LORO
from sklearn.model_selection import LeaveOneGroupOut

logo = LeaveOneGroupOut()
n_folds = logo.get_n_splits(groups=groups)
if n_folds < 2:
    raise RuntimeError(f"Not enough run groups for LORO for {subject}.")

In [317]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold

c = 1.0
class_weight = "balanced"
dual = "auto"
max_iter = 200000

#feature selection 350 was the best for lss, 300 for window
n_select = 300

clf = LinearSVC(C=c, class_weight=class_weight, dual=dual, max_iter=max_iter, random_state=RANDOM_SEED)
pipe = Pipeline(steps=[
    ("var", VarianceThreshold()),
    ("scale", StandardScaler()),
    ("select", SelectKBest(f_classif, k=n_select)),
    ("clf", clf),
])

#store the values of params in csv for comp.
model_class = type(clf).__name__
hyperparams = {**clf.get_params(), "select_k": n_select}

In [318]:
y_pred = np.empty_like(y)
fold_acc, fold_bacc, left_out_runs = [], [], []

In [319]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score

for fold_i, (train_idx, test_idx) in enumerate(logo.split(X_features, y, groups), 1):
    left_out = int(np.unique(groups[test_idx])[0])
    left_out_runs.append(left_out)
    fold_pipe = clone(pipe)
    fold_pipe.fit(X_features[train_idx], y[train_idx])
    y_pred[test_idx] = fold_pipe.predict(X_features[test_idx])
    joblib.dump(fold_pipe, subject_models_dir / f"fold_{fold_i}_pipe.joblib")
    acc = accuracy_score(y[test_idx], y_pred[test_idx])
    bacc = balanced_accuracy_score(y[test_idx], y_pred[test_idx])
    fold_acc.append(acc)
    fold_bacc.append(bacc)
    n_correct = int(np.sum(y[test_idx] == y_pred[test_idx]))
    print(f"Fold {fold_i} (left-out run {left_out}): {n_correct}/{len(test_idx)} correct")
    print(f"Accuracy: {acc:.4f}   Balanced accuracy: {bacc:.4f}")

Fold 1 (left-out run 1): 12/40 correct
Accuracy: 0.3000   Balanced accuracy: 0.2679
Fold 2 (left-out run 2): 14/40 correct
Accuracy: 0.3500   Balanced accuracy: 0.3051
Fold 3 (left-out run 3): 18/40 correct
Accuracy: 0.4500   Balanced accuracy: 0.4464
Fold 4 (left-out run 4): 16/40 correct
Accuracy: 0.4000   Balanced accuracy: 0.2917


In [320]:
fold_acc = np.asarray(fold_acc)
fold_bacc = np.asarray(fold_bacc)

In [321]:
import pandas as pd

fold_balanced_accuracy_df = pd.DataFrame({
    "subject": subject, 
    "fold": np.arange(1, len(fold_bacc) + 1),
    "left_out_run": left_out_runs, 
    "accuracy": fold_acc, 
    "balanced_accuracy": fold_bacc,
})
fold_balanced_accuracy_df.to_csv(subject_results_dir / "fold_balanced_accuracy.csv", index=False)

In [322]:
mean_acc, std_acc = float(fold_acc.mean()), float(fold_acc.std())
mean_bacc, std_bacc = float(fold_bacc.mean()), float(fold_bacc.std())
print(f"Mean accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Mean balanced accuracy: {mean_bacc:.4f} ± {std_bacc:.4f}")
print(f"Chance level: {chance_level:.4f}")

Mean accuracy: 0.3750 ± 0.0559
Mean balanced accuracy: 0.3278 ± 0.0698
Chance level: 0.2500


In [323]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, y_pred, labels=unique_classes)
np.save(subject_results_dir / "confusion_matrix.npy", cm)
row_sum = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, np.where(row_sum == 0, 1, row_sum))
per_class_acc = np.diag(cm_norm)

### Fit data for SVM weight map

In [324]:
full_pipe = Pipeline(steps=[
    ("var", VarianceThreshold()),
    ("scale", StandardScaler()),
    ("select", SelectKBest(f_classif, k=n_select)),
    ("clf", clone(clf)),
])
full_pipe.fit(X_features, y)
joblib.dump(full_pipe, subject_models_dir / "full_pipe.joblib")

#weights in the original space (2187 voxels)
coef_sel = full_pipe.named_steps["select"].inverse_transform(full_pipe.named_steps["clf"].coef_)
coef_full = full_pipe.named_steps["var"].inverse_transform(coef_sel)
print("Saved full_pipe.joblib")

Saved full_pipe.joblib


### Summary metrics for subject

In [325]:
import json

config_key = {"subject": subject, 
              "model_class": model_class,
              "feature_method": FEATURE_METHOD,
              **{f"hp_{k}": v for k, v in hyperparams.items()}
            }

def make_config_id(d):
    return json.dumps(
        {k: (float(v) if isinstance(v, (int, float, np.number)) and not isinstance(v, bool) else str(v))
        for k, v in d.items()},
        sort_keys=True,
    )
config_id = make_config_id(config_key)

summary_row = {
    "config_id": config_id,
    **config_key,
    "n_samples": int(len(y)), 
    "n_voxels": int(X_features.shape[1]),
    "n_classes": int(len(unique_classes)), 
    "mean_accuracy": mean_acc, 
    "std_accuracy": std_acc,
    "mean_balanced_accuracy": mean_bacc, 
    "std_balanced_accuracy": std_bacc,
    "chance_level": chance_level, 
    "tr": tr_subject,
    "samples_per_class": str({c: int(np.sum(y == c)) for c in unique_classes}),
    "mean_per_class_recall": float(np.mean(per_class_acc)),
}

summary_csv = RESULTS_DIR / "svc_multi_subject_summary.csv"
if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    if "config_id" in summary_df.columns:
        summary_df = summary_df[summary_df["config_id"] != config_id]
    summary_df = pd.concat([summary_df, pd.DataFrame([summary_row])], ignore_index=True)
else:
    summary_df = pd.DataFrame([summary_row])

summary_df = summary_df.sort_values(["subject", "model_class", "hp_C"]).reset_index(drop=True)
summary_df.to_csv(summary_csv, index=False)
pd.set_option("display.max_columns", None)
summary_df

,config_id,subject,model_class,feature_method,hp_C,hp_class_weight,hp_dual,hp_fit_intercept,hp_intercept_scaling,hp_loss,hp_max_iter,hp_multi_class,hp_penalty,hp_random_state,hp_tol,hp_verbose,hp_select_k,n_samples,n_voxels,n_classes,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,chance_level,tr,samples_per_class,mean_per_class_recall
0,"{""feature_method"": ""lss"", ""hp_C"": 1.0, ""hp_cla...",sub-p0002,LinearSVC,lss,1.0,balanced,auto,True,1,squared_hinge,200000,ovr,l2,42,0.0001,0,350,160,2187,4,0.36250,0.064952,0.368304,0.050873,0.25,2.0,"{0: 24, 1: 32, 2: 48, 3: 56}",0.368304
1,"{""feature_method"": ""window"", ""hp_C"": 1.0, ""hp_...",sub-p0002,LinearSVC,window,1.0,balanced,auto,True,1,squared_hinge,200000,ovr,l2,42,0.0001,0,350,160,2187,4,0.35000,0.046771,0.346726,0.049723,0.25,2.0,"{0: 24, 1: 32, 2: 48, 3: 56}",0.346726
2,"{""feature_method"": ""window"", ""hp_C"": 1.0, ""hp_...",sub-p0002,LinearSVC,window,1.0,balanced,auto,True,1,squared_hinge,200000,ovr,l2,42,0.0001,0,300,160,2187,4,0.37500,0.053033,0.412574,0.043771,0.25,2.0,"{0: 24, 1: 32, 2: 48, 3: 56}",0.412574
3,"{""feature_method"": ""window"", ""hp_C"": 1.0, ""hp_...",sub-p0004,LinearSVC,window,1.0,balanced,auto,True,1,squared_hinge,200000,ovr,l2,42,0.0001,0,300,160,2187,4,0.41250,0.073951,0.416667,0.060383,0.25,2.0,"{0: 24, 1: 32, 2: 48, 3: 56}",0.416667
4,"{""feature_method"": ""lss"", ""hp_C"": 1.0, ""hp_cla...",sub-p0004,LinearSVC,lss,1.0,balanced,auto,True,1,squared_hinge,200000,ovr,l2,42,0.0001,0,350,160,2187,4,0.38125,0.044634,0.328125,0.038309,0.25,2.0,"{0: 24, 1: 32, 2: 48, 3: 56}",0.328125
5,"{""feature_method"": ""lss"", ""hp_C"": 1.0, ""hp_cla...",sub-p0004,LinearSVC,lss,1.0,balanced,auto,True,1,squared_hinge,200000,ovr,l2,42,0.0001,0,300,160,2187,4,0.37500,0.055902,0.327753,0.069801,0.25,2.0,"{0: 24, 1: 32, 2: 48, 3: 56}",0.327753
